# **Lab Transactions Data Cleaning**

## 1. Importing Libraries and Loading Data

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import sys
sys.path.append(str(Path.cwd().parent))

from scripts.paths import RAW_DIR, CLEANED_DIR

df = pd.read_csv(RAW_DIR / "10_lab_transactions_dirty.csv")

# Load master/reference tables used for data integrity checks.
encounters_df = pd.read_csv(CLEANED_DIR / "05_encounters_cleaned.csv")
lab_tests_df = pd.read_csv(CLEANED_DIR / "09_lab_tests_cleaned.csv")

df.head()

,Lab_Order_ID,Patient_ID,Encounter_ID,Lab_Test_ID,Order_Date,Result_Status,Actual_Cost,Result_Hours
0,LO700000,P105820,E204533,T600078,2025-07-06,Pending,6988.66,24
1,LO700001,P105243,E201482,T600019,2025-09-22,Pending,2375.85,12
2,LO700002,P104745,E202504,T600024,2026-04-24,Completed,11865.12,12
3,LO700003,P100778,E205786,T600002,2025-06-13,NaN,9153.26,72
4,LO700004,P107819,E208710,T600003,2025-03-31,Completed,5567.95,96


## 2. Initial Data Inspection

In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isna().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (18000, 8)

Columns:
['Lab_Order_ID', 'Patient_ID', 'Encounter_ID', 'Lab_Test_ID', 'Order_Date', 'Result_Status', 'Actual_Cost', 'Result_Hours']

Data Types:
Lab_Order_ID      object
Patient_ID        object
Encounter_ID      object
Lab_Test_ID       object
Order_Date        object
Result_Status     object
Actual_Cost      float64
Result_Hours       int64
dtype: object

Missing Values:
Lab_Order_ID        0
Patient_ID          0
Encounter_ID        0
Lab_Test_ID         0
Order_Date          0
Result_Status    3604
Actual_Cost        45
Result_Hours        0
dtype: int64

Duplicate Rows: 0


In [4]:
print("Unique values per column:")
for col in df.columns:
    print(f"{col}: {df[col].nunique(dropna=False)}")

Unique values per column:
Lab_Order_ID: 18000
Patient_ID: 7129
Encounter_ID: 10463
Lab_Test_ID: 140
Order_Date: 601
Result_Status: 5
Actual_Cost: 17795
Result_Hours: 10


## 3. Audit of Specific Columns

In [5]:
# Lab_Order_ID audit
print("Lab_Order_ID duplicate count:", df["Lab_Order_ID"].duplicated().sum())
print("Invalid Lab_Order_ID format:",
      (~df["Lab_Order_ID"].astype("string").str.fullmatch(r"LO\d{6}", na=False)).sum())

Lab_Order_ID duplicate count: 0
Invalid Lab_Order_ID format: 0


In [6]:
# Patient_ID audit
print("Patient_ID missing count:", df["Patient_ID"].isna().sum())
print("Invalid Patient_ID format:",
      (~df["Patient_ID"].astype("string").str.fullmatch(r"P\d{6}", na=False)).sum())

patient_lookup = encounters_df.set_index("Encounter_ID")["Patient_ID"]
expected_patient_id = df["Encounter_ID"].map(patient_lookup)

print("Encounter_ID values not found in encounters master:",
      expected_patient_id.isna().sum())

print("Patient_ID mismatches against Encounter_ID master:",
      (expected_patient_id.notna() & (df["Patient_ID"] != expected_patient_id)).sum())

Patient_ID missing count: 0
Invalid Patient_ID format: 0
Encounter_ID values not found in encounters master: 0
Patient_ID mismatches against Encounter_ID master: 17999


In [7]:
# Encounter_ID audit
print("Encounter_ID missing count:", df["Encounter_ID"].isna().sum())
print("Invalid Encounter_ID format:",
      (~df["Encounter_ID"].astype("string").str.fullmatch(r"E\d{6}", na=False)).sum())
print("Encounter_ID values not found in encounters master:",
      (~df["Encounter_ID"].isin(encounters_df["Encounter_ID"])).sum())

Encounter_ID missing count: 0
Invalid Encounter_ID format: 0
Encounter_ID values not found in encounters master: 0


In [8]:
# Lab_Test_ID audit
print("Lab_Test_ID missing count:", df["Lab_Test_ID"].isna().sum())
print("Invalid Lab_Test_ID format:",
      (~df["Lab_Test_ID"].astype("string").str.fullmatch(r"T\d{6}", na=False)).sum())
print("Lab_Test_ID values not found in lab test master:",
      (~df["Lab_Test_ID"].isin(lab_tests_df["Lab_Test_ID"])).sum())

Lab_Test_ID missing count: 0
Invalid Lab_Test_ID format: 0
Lab_Test_ID values not found in lab test master: 0


In [9]:
# Order_Date audit
order_date_temp = pd.to_datetime(df["Order_Date"], errors="coerce")

print("Order_Date missing count:", df["Order_Date"].isna().sum())
print("Order_Date invalid/unparseable count:", order_date_temp.isna().sum())

print("Minimum Order_Date:", order_date_temp.min())
print("Maximum Order_Date:", order_date_temp.max())

Order_Date missing count: 0
Order_Date invalid/unparseable count: 0
Minimum Order_Date: 2025-01-01 00:00:00
Maximum Order_Date: 2026-08-24 00:00:00


In [10]:
# Result_Status audit
print("Result_Status missing count:", df["Result_Status"].isna().sum())
print("Raw Result_Status values:")
print(df["Result_Status"].value_counts(dropna=False))

status_cleaned = df["Result_Status"].astype("string").str.strip().str.title()
print("\nResult_Status values after formatting audit:")
print(status_cleaned.value_counts(dropna=False))

Result_Status missing count: 3604
Raw Result_Status values:
Result_Status
Completed     3630
Pending       3622
NaN           3604
completed     3584
Cancelled     3560
Name: count, dtype: int64

Result_Status values after formatting audit:
Result_Status
Completed    7214
Pending      3622
<NA>         3604
Cancelled    3560
Name: count, dtype: Int64


In [11]:
# Actual_Cost audit
actual_cost_numeric = pd.to_numeric(df["Actual_Cost"], errors="coerce")

print("Actual_Cost missing/non-numeric count:", actual_cost_numeric.isna().sum())
print("Actual_Cost <= 0 count:", (actual_cost_numeric <= 0).sum())
print("Actual_Cost < 0 count:", (actual_cost_numeric < 0).sum())
print("Actual_Cost == 0 count:", (actual_cost_numeric == 0).sum())

print("\nActual_Cost summary:")
print(actual_cost_numeric.describe())

Actual_Cost missing/non-numeric count: 45
Actual_Cost <= 0 count: 85
Actual_Cost < 0 count: 50
Actual_Cost == 0 count: 35

Actual_Cost summary:
count    17955.000000
mean      9087.880893
std       5187.778966
min       -100.000000
25%       4582.525000
50%       9128.040000
75%      13553.825000
max      17997.320000
Name: Actual_Cost, dtype: float64


In [12]:
# Result_Hours audit
result_hours_numeric = pd.to_numeric(df["Result_Hours"], errors="coerce")

print("Result_Hours missing/non-numeric count:", result_hours_numeric.isna().sum())
print("Result_Hours <= 0 count:", (result_hours_numeric <= 0).sum())

print("\nUnique Result_Hours values:")
print(sorted(result_hours_numeric.dropna().unique().tolist()))

Result_Hours missing/non-numeric count: 0
Result_Hours <= 0 count: 0

Unique Result_Hours values:
[1, 2, 5, 8, 12, 24, 36, 48, 72, 96]


## 4. Data Cleaning

### 4.1 Patient_ID

In [13]:
# Use the Encounter_ID master relationship to repair inconsistent Patient_ID values.
patient_lookup = encounters_df.set_index("Encounter_ID")["Patient_ID"]
patient_cleaned = df["Patient_ID"].copy()

mapped_patient_id = df["Encounter_ID"].map(patient_lookup)

patient_cleaned = patient_cleaned.mask(
    mapped_patient_id.notna(),
    mapped_patient_id
)

df["Patient_ID"] = patient_cleaned

### 4.2 Result_Status

In [14]:
result_status_cleaned = (
    df["Result_Status"]
    .astype("string")
    .str.strip()
    .str.title()
)

df["Result_Status"] = result_status_cleaned

### 4.3 Order_Date

In [15]:
order_date_cleaned = pd.to_datetime(
    df["Order_Date"],
    errors="coerce"
)

df["Order_Date"] = order_date_cleaned

### 4.4 Actual_Cost

In [16]:
actual_cost_cleaned = pd.to_numeric(
    df["Actual_Cost"],
    errors="coerce"
)

# Actual_Cost must be positive. Invalid non-positive values become missing.
actual_cost_cleaned = actual_cost_cleaned.mask(
    actual_cost_cleaned <= 0
)

df["Actual_Cost"] = actual_cost_cleaned.astype("Float64")

### 4.5 Result_Hours

In [17]:
# Result_Hours contains only positive numeric values, so no transformation is required.
# Keep the original values unchanged.
df["Result_Hours"] = pd.to_numeric(
    df["Result_Hours"],
    errors="coerce"
).astype("Int64")

## 5. Final Validation

In [18]:
print("Shape:", df.shape)
print("\nDuplicate Rows:", df.duplicated().sum())
print("Lab_Order_ID duplicates:", df["Lab_Order_ID"].duplicated().sum())

print("\nMissing values:")
print(df.isna().sum())

print("\nData Types:")
print(df.dtypes)

Shape: (18000, 8)

Duplicate Rows: 0
Lab_Order_ID duplicates: 0

Missing values:
Lab_Order_ID        0
Patient_ID          0
Encounter_ID        0
Lab_Test_ID         0
Order_Date          0
Result_Status    3604
Actual_Cost       130
Result_Hours        0
dtype: int64

Data Types:
Lab_Order_ID             object
Patient_ID               object
Encounter_ID             object
Lab_Test_ID              object
Order_Date       datetime64[ns]
Result_Status    string[python]
Actual_Cost             Float64
Result_Hours              Int64
dtype: object


In [19]:
# Validate Patient_ID after cleaning
expected_patient_id = df["Encounter_ID"].map(
    encounters_df.set_index("Encounter_ID")["Patient_ID"]
)

patient_mismatch_mask = (
    df["Patient_ID"].notna()
    & expected_patient_id.notna()
    & (df["Patient_ID"] != expected_patient_id)
)

print("Patient_ID mismatches after cleaning:",
      patient_mismatch_mask.sum())

# Validate Lab_Test_ID
invalid_lab_test_mask = (
    df["Lab_Test_ID"].notna()
    & ~df["Lab_Test_ID"].isin(
        set(lab_tests_df["Lab_Test_ID"])
    )
)

print("Invalid Lab_Test_ID after cleaning:",
      invalid_lab_test_mask.sum())

# Validate Result_Status
print("Result_Status values after cleaning:")
print(df["Result_Status"].value_counts(dropna=False))

Patient_ID mismatches after cleaning: 0
Invalid Lab_Test_ID after cleaning: 0
Result_Status values after cleaning:
Result_Status
Completed    7214
Pending      3622
<NA>         3604
Cancelled    3560
Name: count, dtype: Int64


In [20]:
print(
    "Invalid Order_Date after cleaning:",
    df["Order_Date"].isna().sum()
)

print(
    "Non-positive Actual_Cost after cleaning:",
    (df["Actual_Cost"].dropna() <= 0).sum()
)

print(
    "Non-positive Result_Hours after cleaning:",
    (df["Result_Hours"].dropna() <= 0).sum()
)

print(
    "Invalid Patient_ID format after cleaning:",
    (
        ~df["Patient_ID"]
        .astype("string")
        .str.fullmatch(r"P\d{6}", na=False)
    ).sum()
)

print(
    "Invalid Lab_Order_ID format after cleaning:",
    (
        ~df["Lab_Order_ID"]
        .astype("string")
        .str.fullmatch(r"LO\d{6}", na=False)
    ).sum()
)

Invalid Order_Date after cleaning: 0
Non-positive Actual_Cost after cleaning: 0
Non-positive Result_Hours after cleaning: 0
Invalid Patient_ID format after cleaning: 0
Invalid Lab_Order_ID format after cleaning: 0


## 6. Export Cleaned Data

In [22]:
df.to_csv(CLEANED_DIR / "10_lab_orders_cleaned.csv",index=False)